In [ ]:
import pandas as pd
import numpy as np
from tomlkit import date

创建一个包含10名学生数学成绩的Series，成绩范围在50-100之间，计算平均分、最高分、最低分，并找出高于平均分的学生人数

In [ ]:
arr = np.random.randint(50,101,10)
s = pd.Series(arr,index=['学生'+str(i) for i in range(1,11)])
print(s,'\n')
print(s.max())
print(s.min())
print(s.mean())
print(s[s.values>s.mean()].count())

给定一个城市每天的最高温度Series，完成以下任务

找出温度高于30的天数

计算平均温度

温度从高到低排序

找出温度变化最大的两天


In [ ]:
ind = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
tempe = np.random.randint(20,41,7)
s = pd.Series(tempe,index=ind)
print(s,'\n')
print('温度高于30的天数:',s[s.values>30].count())
print('平均气温:',s.mean())
sort_s = s.sort_values(ascending=False).index.tolist() #tolist()去掉index dtype
print('排序:',*(sort_s)) #*去掉中括号

changes = list()
d = list()

for i in range(len(ind)-1):
    changes.append(abs(tempe[i+1]-tempe[i]))

for j in range(len(ind)-1):
    d.append(ind[j+1]+','+ind[j])

s0 = pd.Series(changes,index=d)
print(s0.idxmax())
#print(s0[s0.values==s0.max()].keys())

tempe0 = s.diff().abs()
print(tempe0)

给定股票连续10个交易日的收盘价：

计算每日收益率

找出收益率最高最低日期

计算波动率

In [ ]:
dates = pd.date_range('2026-05-18', periods=10, freq='B') # B代表工作日 自动跳过周末
prices = [100, 102, 101, 105, 104, 108, 110, 109, 112, 115]
s = pd.Series(prices,index=dates)

pp = [np.nan]
for i in range(1,len(dates)):
    pp.append((prices[i]/prices[i-1])-1)

s0 = pd.Series(pp,index=dates,name='收益率') #pp = s.pct_change() 计算百分率变化
print(s0)

print('最高:',s0.idxmax())
print('最低:',s0.idxmin())

print('波动率:',s0.std())

某产品过去12个月的销售量Series：

计算季度平均销量（每3个月为一个季度）

找出销量最高的月份

计算月环比增长率

找出连续增长超过2个月的月份

In [ ]:
dates = pd.date_range('2025-01-01', periods=12, freq='MS') # MS代表每月第一天 ME代表每月最后一天
sales = pd.Series([
    120, 135, 180,
    210, 190, 160,
    110, 105, 140,
    165, 195, 220
], index=dates, name='销量')

print(sales.resample('QS').mean()) #resample重新采样

print('销量最多为%d月'%sales.idxmax().month)

print(sales.pct_change())

temp = sales.pct_change()
mask = temp.rolling(window=3, min_periods=3).min() > 0 #默认min_period=windows
#min_period: 在窗口框定的存在元素（去掉nan）个数达到min_period的时候窗口才返回这个切片
#rolling()滚动窗口是向前看而非向后看
print(*(temp[mask].keys().month.tolist()))

补充:
D	Calendar day	自然日（包含周末）

B	Business day	工作日（跳过周末）

C	Custom business day	自定义工作日（可指定节假日）

W	Week	周（默认周日结束，可用 W-MON 改为周一结束）

M / ME	Month end	月末（旧版用 M，新版推荐 ME）

MS	Month start	月初（每月1号）

BM / BME	Business month end	月末最后一个工作日

BMS	Business month start	月初第一个工作日

Q / QE	Quarter end	季度末（3/6/9/12月底）

QS	Quarter start	季度初（1/4/7/10月初）

BQ / BQE	Business quarter end	季度末最后一个工作日

BQS	Business quarter start	季度初第一个工作日

A / Y / YE	Year end	年末（12月31日）

AS / YS	Year start	年初（1月1日）

h	Hour	小时

min / T	Minute	分钟

s	Second	秒

ms	Millisecond	毫秒

us	Microsecond	微秒


某商店每小时销售额 Series：

按天重采样计算每日总销售额。

计算每天营业时间（8:00-22:00）和非营业时间的销售额比例。

找出销售额最高的 3 个小时。

In [ ]:
# 1. 生成模拟数据
# 创建一个从 2023-10-01 开始，频率为 1小时 的时间索引，共 72 小时（3天）
dates = pd.date_range(start='2023-10-01', periods=72, freq='h')

# 使用列表推导式生成符合业务逻辑的销售额
# 如果小时数在 8点到22点之间(不含22点)，生成大额销售；否则生成小额销售
sales_data = [
    np.random.randint(100, 500) if 8 <= t.hour < 22 else np.random.randint(0, 50)
    for t in dates
]

# 创建 Series
s = pd.Series(sales_data, index=dates)
s.name = "Hourly_Sales"

print("--- 原始数据预览 ---")
print(s.head(10))
print("...\n")


# 2. 按天重采样计算每日总销售额
# 'D' 代表按天聚合，sum() 代表求和
daily_total = s.resample('D').sum()

print("--- 每日总销售额 ---")
print(daily_total)
print("\n")

# 3. 计算营业时间 vs 非营业时间比例
# 利用 index 的 hour 属性进行布尔筛选
# 注意：8:00-22:00 通常指包含8点整，不包含22点整（即21:00是最后一个营业小时）
is_business_hours = (s.index.hour >= 8) & (s.index.hour < 22)

# 分别求和
business_sales = s[is_business_hours].resample('D').sum()
non_business_sales = s[~is_business_hours].resample('D').sum() #~表示逻辑非

# 计算比例：营业 / 非营业
# 为了防止除以0报错，可以用 fillna(0) 处理没有非营业数据的极端情况
ratio = business_sales / non_business_sales.fillna(1) # 这里填1是为了演示，实际可能填0或忽略
#fillna()，当你的数据中存在 NaN时，你可以用它指定一个值来替换这些空缺

result_ratio = pd.DataFrame({
    '营业总额': business_sales,
    '非营业总额': non_business_sales,
    '销额比例': ratio
})

print("--- 每天营业与非营业销售额比例 ---")
print(result_ratio)
print("\n")

# 4. 找出销售额最高的 3 个小时
# nlargest(n) 是获取前N大最高效的方法 series与dataframe都有
# nsmallest同理
top_3_hours = s.nlargest(3)

print("--- 销售额最高的 3 个小时 ---")
print(top_3_hours)